In [12]:
import os
import json
import re
import spacy

# =========================
# CONFIGURATION
# =========================
WHISPER_MODEL = "base"
AUDIO_FOLDER = r"D:\ai_meeting_intelligence\data"   # 👈 change if needed
SAMPLE_TRANSCRIPTS = {}  # optional fallback

# =========================
# LOAD NLP MODEL
# =========================
nlp = spacy.load("en_core_web_sm")


# =========================
# WHISPER TRANSCRIBER
# =========================
class WhisperTranscriber:

    def __init__(self, model_size='base'):
        self.model_size = model_size
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            import whisper
            print(f'🔄 Loading Whisper model: {self.model_size}...')
            self.model = whisper.load_model(self.model_size)
            print(f'✅ Whisper {self.model_size} model loaded!')
        except:
            print("⚠️ Whisper not available, using fallback text")

    def transcribe(self, audio_path):
        filename = os.path.basename(audio_path)

        if self.model:
            try:
                result = self.model.transcribe(audio_path, fp16=False)
                return result
            except Exception as e:
                print(f"⚠️ Error: {e}")

        return {"text": "No transcript available.", "segments": []}

    def transcribe_all(self, folder):
        results = {}
        extensions = {'.wav', '.mp3', '.m4a', '.flac'}

        files = [f for f in os.listdir(folder)
                 if os.path.splitext(f)[1].lower() in extensions]

        print(f"\n🎙️ Processing {len(files)} files...")

        for i, file in enumerate(files, 1):
            path = os.path.join(folder, file)

            print(f"\n[{i}/{len(files)}] {file}")
            result = self.transcribe(path)

            print(f"📝 {result['text'][:80]}...")
            results[file] = result

        return results


# =========================
# NLP FUNCTIONS
# =========================
def generate_summary(text):
    doc = nlp(text)
    sentences = [s.text.strip() for s in doc.sents]
    return " ".join(sentences[:2])


def extract_actions(text):
    doc = nlp(text)
    actions = []

    for sent in doc.sents:
        sentence = sent.text.strip()

        if any(k in sentence.lower() for k in ["will", "need to", "should", "by"]):

            # Extract PERSON entity
            owner = None
            for ent in doc.ents:
                if ent.label_ == "PERSON":
                    owner = ent.text

            # Extract deadline
            deadline = None
            match = re.search(r"\b(Monday|Tuesday|Wednesday|Thursday|Friday)\b", sentence)
            if match:
                deadline = match.group(0)

            actions.append({
                "task": sentence,
                "owner": owner,
                "deadline": deadline
            })

    return actions


# =========================
# MAIN PIPELINE
# =========================
transcriber = WhisperTranscriber(WHISPER_MODEL)

transcripts = transcriber.transcribe_all(AUDIO_FOLDER)

final_output = []

for fname, data in transcripts.items():
    text = data["text"]

    summary = generate_summary(text)
    actions = extract_actions(text)

    meeting_result = {
        "meeting_file": fname,
        "summary": summary,
        "action_items": actions
    }

    final_output.append(meeting_result)

    print("\n" + "="*50)
    print(f"📁 Meeting: {fname}")
    print("🧾 Summary:", summary)

    print("\n📌 Action Items:")
    for a in actions:
        print(f"- Task: {a['task']}")
        print(f"  Owner: {a['owner']}")
        print(f"  Deadline: {a['deadline']}")


# =========================
# SAVE OUTPUT
# =========================
with open("final_output.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("\n✅ Output saved to final_output.json")
print("📊 Total meetings processed:", len(final_output))

🔄 Loading Whisper model: base...
✅ Whisper base model loaded!

🎙️ Processing 6 files...

[1/6] Council Meeting Audio 2026-4-7.mp3
⚠️ Error: [WinError 2] The system cannot find the file specified
📝 No transcript available....

[2/6] meeting_01_product_launch.wav
⚠️ Error: [WinError 2] The system cannot find the file specified
📝 No transcript available....

[3/6] meeting_02_sprint_planning.wav
⚠️ Error: [WinError 2] The system cannot find the file specified
📝 No transcript available....

[4/6] meeting_03_client_review.wav
⚠️ Error: [WinError 2] The system cannot find the file specified
📝 No transcript available....

[5/6] meeting_04_budget_discussion.wav
⚠️ Error: [WinError 2] The system cannot find the file specified
📝 No transcript available....

[6/6] meeting_05_team_standup.wav
⚠️ Error: [WinError 2] The system cannot find the file specified
📝 No transcript available....

📁 Meeting: Council Meeting Audio 2026-4-7.mp3
🧾 Summary: No transcript available.

📌 Action Items:

📁 Meeting: me

In [ ]:
ffmpeg -version

In [5]:
!ffmpeg -version

'ffmpeg' is not recognized as an internal or external command,
operable program or batch file.
